# 🔮 Notebook 05: Future Forecasting

**Goal**: Forecast from **1 Juni 2023 → 29 Mei 2026** (sesuai instruksi poin 6)

Steps:
1. Select the best model from benchmarking
2. Re-train on full dataset
3. Generate iterative future forecasts
4. Plot: historical actual + forecast
5. Save submission CSV

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import DataLoader
from src.features.feature_engineering import TimeSeriesFeatureEngineer
from src.models.ml_models import XGBoostModel   # ← swap to your best model
from src.visualization.plots import plot_forecast
from benchmarking.forecaster import FutureForecaster

print('✅ Forecasting modules loaded')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────
DATE_COL   = 'date'    # ← UPDATE
TARGET_COL = 'value'   # ← UPDATE
FREQ       = 'D'       # ← 'D'=daily, 'W'=weekly, 'M'=monthly
FORECAST_START = '2023-06-01'
FORECAST_END   = '2026-05-29'

# ── Load full data ─────────────────────────────────────────────────
loader = DataLoader()
df = loader.load_csv('data/raw/train.csv')  # ← UPDATE
print(f'Full dataset: {df.shape}')
df.tail(3)

In [ ]:
# ── Feature Engineering on FULL dataset ───────────────────────────────
feat_eng = TimeSeriesFeatureEngineer(
    target_col=TARGET_COL, date_col=DATE_COL
)
df_fe = feat_eng.fit_transform(df).dropna()
feat_cols = feat_eng.get_feature_columns(df_fe)

X_full = df_fe[feat_cols]
y_full = df_fe[TARGET_COL]

print(f'Full features: {X_full.shape}')

In [ ]:
# ── Train best model on FULL dataset ──────────────────────────────────
best_model = XGBoostModel(params={
    'n_estimators': 300, 'learning_rate': 0.05,
    'max_depth': 6, 'random_state': 42
})
best_model.fit_timed(X_full, y_full)
print(f'✅ Model trained in {best_model.fit_time_:.2f}s')

In [ ]:
# ── Generate Future Forecast ───────────────────────────────────────────
forecaster = FutureForecaster(
    model=best_model,
    feature_engineer=feat_eng,
    target_col=TARGET_COL,
    date_col=DATE_COL,
    freq=FREQ,
)

forecast_df = forecaster.forecast(
    history_df=df,
    start_date=FORECAST_START,
    end_date=FORECAST_END,
)
print(f'Forecast shape: {forecast_df.shape}')
forecast_df.head()

In [ ]:
# ── Plot: Historical + Forecast ────────────────────────────────────────
fig = plot_forecast(
    train_dates=df[DATE_COL],
    train_actual=df[TARGET_COL].values,
    forecast_dates=forecast_df[DATE_COL],
    forecast_values=forecast_df['forecast'].values,
    model_name=best_model.name,
    save_path='results/plots/05_forecast_full.png',
)
plt.show()
print('✅ Forecast plot saved')

In [ ]:
# ── Save submission CSV ────────────────────────────────────────────────
submission_path = forecaster.save_forecast(
    forecast_df,
    path='data/submissions/forecast_submission.csv'
)
print(f'✅ Forecast saved: {submission_path}')
forecast_df.describe()